####  Bronze vs Silver Testing – Transaction Items Table

This document describes the data quality, integrity, and reconciliation tests performed for the `transaction_items` table during ingestion from Bronze to Silver.





####  Tables Under Test

- Bronze: `coffee.bronze.transaction_items`
- Silver: `coffee.silver.transaction_items`
- Quarantine: `coffee.silver.transaction_items_quarantine`

In [0]:
-- Test 1: Record count comparison between Bronze and Silver
-- Silver count may be LESS than Bronze because:
--   1) invalid rows are filtered/quarantined
--   2) exact duplicates are rolled up into one row

SELECT 'bronze' AS layer, COUNT(*) AS record_count
FROM coffee.bronze.transaction_items

UNION ALL

SELECT 'silver' AS layer, COUNT(*) AS record_count
FROM coffee.silver.transaction_items;


In [0]:
-- Test 2: Subtotal reconciliation
-- Even though Silver may have fewer rows due to rollup,
-- the total subtotal should remain the SAME (business metric must not change).

SELECT 'bronze' AS layer, ROUND(SUM(CAST(subtotal AS DOUBLE)), 2) AS total_subtotal
FROM coffee.bronze.transaction_items

UNION ALL

SELECT 'silver' AS layer, ROUND(SUM(subtotal), 2) AS total_subtotal
FROM coffee.silver.transaction_items;


In [0]:
-- Test 3: Quantity reconciliation
-- Silver is a rollup version, so total quantity should remain unchanged.

SELECT 'bronze' AS layer, SUM(CAST(quantity AS INT)) AS total_quantity
FROM coffee.bronze.transaction_items

UNION ALL

SELECT 'silver' AS layer, SUM(quantity) AS total_quantity
FROM coffee.silver.transaction_items;


In [0]:
-- Test 4: Surrogate key uniqueness check
-- transaction_item_sk must be UNIQUE in Silver

SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT transaction_item_sk) AS distinct_sk,
  (COUNT(*) - COUNT(DISTINCT transaction_item_sk)) AS duplicate_sk_rows
FROM coffee.silver.transaction_items;


In [0]:
-- Test 5: Validate rollup for a specific transaction_id


SELECT *
FROM coffee.silver.transaction_items
WHERE transaction_id = '0fb3cba6-327a-4146-98c8-c3dffec64236'
ORDER BY created_at;


In [0]:
-- same transaction_id for bronze to show rollling up in silver
SELECT *
FROM coffee.bronze.transaction_items
WHERE transaction_id = '0fb3cba6-327a-4146-98c8-c3dffec64236'
ORDER BY created_at;


In [0]:
describe table coffee.silver.transaction_items